In [ ]:
!pip install lightgbm

## Идея: Смешение расчетного значения SI и предсказанного независимой моделью
Почему отказались: поняли, что такой подход расценивается как утечка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from lightgbm import LGBMRegressor

In [2]:
# Загружаем данные
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

In [3]:
IC50_PHYSICAL_MIN = train['IC50, mM'].min() # 0.003517

In [4]:
target_cols_all = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in test.columns if col not in target_cols_all and col != 'index']

X_train_raw = train[feature_cols].fillna(train[feature_cols].median())
X_test_raw = test[feature_cols].fillna(train[feature_cols].median())


In [5]:
submission_final = pd.DataFrame({'index': test['index']})
kf = KFold(n_splits=5, shuffle=True, random_state=42)

pure_test_preds = {}

In [6]:
# Обучаем модели на ПОЛНОМ датасете, чтобы вернуть крупные числа
for target in target_cols_all:
    y_train = train[target]
    
    # Отбор топ-95 признаков индивидуально
    selector = SelectKBest(score_func=f_regression, k=95)
    X_train_selected = pd.DataFrame(selector.fit_transform(X_train_raw, y_train))
    X_test_selected = pd.DataFrame(selector.transform(X_test_raw))
    
    test_preds = np.zeros(len(test))
    rmse_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_selected, y_train)):
        X_tr, y_tr = X_train_selected.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train_selected.iloc[val_idx], y_train.iloc[val_idx]
        
        model = CatBoostRegressor(
            iterations=1800,       # Увеличим число деревьев для фиксации редких связей
            learning_rate=0.012,   # Уменьшим шаг для точности
            depth=6,               # Чуть глубже, чтобы поймать крупные выбросы
            loss_function='RMSE',
            l2_leaf_reg=5,
            random_seed=42,
            verbose=False
        )
        
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=100, verbose=False)
        val_preds = model.predict(X_va)
        rmse_scores.append(root_mean_squared_error(y_va, val_preds))
        
        test_preds += model.predict(X_test_selected) / kf.n_splits
        
    print(f"Полный CatBoost [{target}] -> Локальный CV RMSE: {np.mean(rmse_scores):.4f}")
    pure_test_preds[target] = test_preds

Полный CatBoost [IC50, mM] -> Локальный CV RMSE: 313.6194
Полный CatBoost [CC50, mM] -> Локальный CV RMSE: 436.1185
Полный CatBoost [SI] -> Локальный CV RMSE: 592.7898


In [7]:
# Шаг 2: Формируем базовые концентрации с физическим ограничением снизу
submission_final['IC50'] = np.clip(pure_test_preds['IC50, mM'], 0.0, None)
submission_final['CC50'] = np.clip(pure_test_preds['CC50, mM'], 0.0, None)

In [8]:
# Шаг 3: ВЗВЕШЕННЫЙ БЛЕНДИНГ СТРАТЕГИЙ ДЛЯ SI
# Стратегия 1: Чистый математический расчет (давал лучший базовый Score 282)
protected_ic50 = np.clip(submission_final['IC50'], IC50_PHYSICAL_MIN, None)
si_strategy_math = submission_final['CC50'] / protected_ic50

# Стратегия 2: Прямое предсказание на полном масштабе (возвращает пики выбросов)
si_strategy_model = np.clip(pure_test_preds['SI'], 0.0, None)

# Смешиваем их: 75% стабильной математики + 25% смелых пиковых предсказаний модели
# Это защитит нас от занижения крупных молекул в тесте
submission_final['SI'] = 0.75 * si_strategy_math + 0.25 * si_strategy_model

print("\nПроверка средних значений финального ансамбля:")
print(submission_final[['IC50', 'CC50', 'SI']].mean())

# Запись
submission_final.to_csv('absolute_blend_submission.csv', index=False)
print("\n[ГОТОВО] Файл 'absolute_blend_submission.csv' успешно создан.")


Проверка средних значений финального ансамбля:
IC50    241.124152
CC50    648.036837
SI       24.716853
dtype: float64

[ГОТОВО] Файл 'absolute_blend_submission.csv' успешно создан.
